# Dos parches a la vez: ¿cada uno sigue funcionando con el otro prendido? (Kaggle, T4 x2)

Modelo **Qwen3.5-0.8B**, dos `.moipatch`:
- **loop:** el arreglo p50c (capas 15 y 19, `down_proj`), ya en `patches/`.
- **anti-sicofancia:** se crea aquí mismo para el 0.8B, con el mismo método que el del 4B (v3: dirección presión-vs-neutral + compuerta de confianza).

**Qué mide:**
1. **Sicofancia** (1000 preguntas MMLU, probabilidad de cada letra, sin sorteo): sostuvo / aceptó / neutral / turno 1 con base, solo sicofancia, solo loop prendido, y **los dos** (loop prendido + sicofancia con su router y su compuerta leídos del modelo ya con el loop).
2. **Loops** (200 GSM8K a 16k, vLLM fp16): base, loop siempre prendido, y **los dos siempre prendidos** (el peor caso).
3. **Routers:** ¿el router de sicofancia se dispara en problemas de mate, donde debería quedarse apagado?
4. Si los dos parches tocan **la misma matriz**, también prueba el otro orden y una versión ortogonalizada.

**Uso:** Settings → **Accelerator: GPU T4 x2**, **Internet: On** → **Save Version → Save & Run All (Commit)**. ~3 h. Resultado en el **Output**: `results_moi_compose.json` y el parche nuevo `qwen35_0.8b_anti_sycophancy.moipatch`.

In [ ]:
import os, json, subprocess, sys, time
REPO = "/kaggle/working/wsi"
BASE = "/tmp/qwen-base"
SYC = "/tmp/syc"                   # pasos intermedios grandes (no van al Output)
GSM = "/tmp/gsm"
OUTD = "/kaggle/working"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q vllm
import vllm, transformers, torch; print("vllm", vllm.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
assert os.path.exists(f"{REPO}/scripts/moi_compose.py"), "moi_compose.py no está en GitHub todavía: falta hacer push"
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3.5-0.8B", local_dir=BASE)
print("commit", COMMIT)

In [ ]:
# 1. parche anti-sicofancia para el 0.8B (mismo método que el del 4B, v3). ~1 h
GATES = "attn:top@8,attn:top@16,attn:late@4,attn:late@8,attn:fa@4,mlpall:top@8,mlpall:top@16"
SYC_RES = f"{OUTD}/results_moi_sycophancy_08b.json"
!python -u {REPO}/scripts/moi_sycophancy.py --model {BASE} --work {SYC} --out {SYC_RES} --dtype fp32 --direction pressure --alphas 2,4,8,16 --gate-configs {GATES} 2>&1 | grep --line-buffered -v -E "Loading weights|fast path|Warning|Generating .* split"

In [ ]:
# 2. empaquetar como .moipatch: la mejor fila de dev que se pueda empaquetar (compuerta de confianza o sin compuerta)
r = json.load(open(SYC_RES))
rows = r["gate_study"]["rows"]
ok = {k: v for k, v in rows.items() if k.split("|")[1] in ("maxprob", "ungated")}
CFG = max(ok, key=lambda k: ok[k]["dev_score"])
print("config:", CFG, "| test held", ok[CFG]["held"], "accepted", ok[CFG]["accepted"], "| base", r["gate_study"]["base_test"])
SYC_PATCH = f"{OUTD}/qwen35_0.8b_anti_sycophancy.moipatch"
!python {REPO}/scripts/make_moipatch_sycophancy.py --work {SYC} --results {SYC_RES} --model-dir {BASE} --config "{CFG}" --base-model Qwen/Qwen3.5-0.8B --out {SYC_PATCH}
LOOP_PATCH = f"{REPO}/patches/qwen35_0.8b_p50c_loop_fix.moipatch"

In [ ]:
# 3. sicofancia con el loop prendido (MMLU) + ¿el router de sicofancia se dispara en mate?
!python {REPO}/scripts/moi_compose.py prepare --work {GSM} --base-dir {BASE}
!python -u {REPO}/scripts/moi_compose.py syc --model {BASE} --syc-work {SYC} --syc-patch {SYC_PATCH} --loop-patch {LOOP_PATCH} --gsm-requests {GSM}/requests_test.jsonl --dtype fp32 --out {OUTD}/compose_syc.json 2>&1 | grep --line-buffered -v -E "Loading weights|fast path|Warning"

In [ ]:
# 4. loops en GSM8K (vLLM fp16): base y loop en paralelo (una T4 cada uno), luego los dos parches juntos
!python {REPO}/scripts/moi_compose.py bake --base-dir {BASE} --out-dir /tmp/ck-loop --loop-patch {LOOP_PATCH}
!python {REPO}/scripts/moi_compose.py bake --base-dir {BASE} --out-dir /tmp/ck-both --loop-patch {LOOP_PATCH} --syc-patch {SYC_PATCH} --mode seq

def gen(model, out, gpu):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    log = open(f"/tmp/gen_{out}.log", "w")
    return subprocess.Popen([sys.executable, f"{REPO}/scripts/moi_compose.py", "gen", "--work", GSM, "--model", model,
                             "--out", f"outputs_test_{out}.jsonl", "--dtype", "float16"], env=env, stdout=log, stderr=subprocess.STDOUT)

def wait(ps):
    for name, p in ps:
        rc = p.wait()
        print(name, "rc", rc); print(open(f"/tmp/gen_{name}.log").read()[-1500:])
        assert rc == 0, f"falló la generación {name}"

ngpu = torch.cuda.device_count()
todo = [("base", BASE), ("loop", "/tmp/ck-loop"), ("both", "/tmp/ck-both")]
todo = [(n, m) for n, m in todo if not os.path.exists(f"{GSM}/outputs_test_{n}.jsonl")]
while todo:
    batch, todo = todo[:ngpu], todo[ngpu:]
    wait([(n, gen(m, n, i)) for i, (n, m) in enumerate(batch)])

In [ ]:
# 5. reporte (va al Output)
!python {REPO}/scripts/moi_compose.py report --work {GSM} --syc-json {OUTD}/compose_syc.json --vllm-dtype float16 --out results_moi_compose.json
import shutil
res = json.load(open(f"{GSM}/results_moi_compose.json"))
res["_final"] = {"complete": True, "commit": COMMIT, "syc_config": CFG, "notebook": "moi_compose_kaggle.ipynb"}
with open(f"{OUTD}/results_moi_compose.json.tmp", "w") as f:
    json.dump(res, f, indent=1)
os.replace(f"{OUTD}/results_moi_compose.json.tmp", f"{OUTD}/results_moi_compose.json")
shutil.rmtree(REPO, ignore_errors=True)    # el Output solo lleva los resultados y el parche
print(sorted(os.listdir(OUTD)))